# Notebook 03 — System Implications

**Purpose:** Translate the conditional Monte Carlo commitment depths from
notebook 02 into accredited MW, avoided capacity cost, and interconnection
acceleration NPV. Reproduces Nature Energy manuscript §6.

**Inputs** (from notebook 02):
- `outputs/contracts/cascade_parameters.json`
- `outputs/contracts/conditional_mc_results.json`

**Outputs:**
- `outputs/contracts/final_results.json`
- `outputs/tables/system_implications_summary.csv`

---

## Notebook Architecture

| Part | Section | Contents |
|---|---|---|
| **0** | Setup | Imports, contract loading, sanity checks |
| **1** | Accredited Capacity (§6) | ELCC application, reference cases, per-GW curve |
| **2** | Avoided Capacity Cost (§6) | E3 levelized CT, CEJA sensitivity |
| **3** | Interconnection Acceleration NPV (§6) | Foregone revenue, sensitivity grid |
| **4** | Summary and Exports | Consolidated table, validation, final_results.json |

## Part 0: Setup

- **0.1** Imports and configuration
- **0.2** Load contracts from notebook 02
- **0.3** Contract sanity checks

In [1]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 0-1: IMPORTS AND CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════
import numpy as np
import pandas as pd
import json
import os
from pathlib import Path

# ─── REPO_ROOT resolver (same pattern as notebooks 01 and 02) ────────────────
_cwd = Path.cwd()
if (_cwd / 'notebooks').is_dir():
    REPO_ROOT = _cwd
elif _cwd.name == 'notebooks' and (_cwd.parent / 'notebooks').is_dir():
    REPO_ROOT = _cwd.parent
else:
    REPO_ROOT = _cwd

CONTRACTS_DIR = REPO_ROOT / 'outputs' / 'contracts'
TABLES_DIR    = REPO_ROOT / 'outputs' / 'tables'
FIGURES_DIR   = REPO_ROOT / 'outputs' / 'figures'

os.makedirs(CONTRACTS_DIR, exist_ok=True)
os.makedirs(TABLES_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

print(f"REPO_ROOT:     {REPO_ROOT}")
print(f"CONTRACTS_DIR: {CONTRACTS_DIR}")

REPO_ROOT:     C:\Users\dunla\repos\data-center-flexibility-resource-adequacy
CONTRACTS_DIR: C:\Users\dunla\repos\data-center-flexibility-resource-adequacy\outputs\contracts


### 0.2 Load Contracts from Notebook 02

In [2]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 0-2: LOAD CONTRACTS FROM NOTEBOOK 02
# ══════════════════════════════════════════════════════════════════════════════

with open(CONTRACTS_DIR / 'cascade_parameters.json') as f:
    cascade_params = json.load(f)

with open(CONTRACTS_DIR / 'conditional_mc_results.json') as f:
    conditional_mc = json.load(f)

# Reformed framework: schema v2.0, two scope scenarios reported in parallel
assert cascade_params.get('version') == '2.0', (
    f"Expected cascade_parameters schema v2.0; got {cascade_params.get('version')}. "
    "Notebook 02 must be re-run under reformed framework."
)
assert conditional_mc.get('version') == '2.0', (
    f"Expected conditional_mc_results schema v2.0; got {conditional_mc.get('version')}. "
    "Notebook 02 must be re-run under reformed framework."
)

print(f"cascade_parameters.json (v{cascade_params['version']}):")
print(f"  Mixed-use static commit:        "
      f"{cascade_params['static_commitment_depth_central']['mixed_use']:.1%}")
print(f"  Inference-dominant static commit: "
      f"{cascade_params['static_commitment_depth_central']['inference_dominant']:.1%}")
print()
print(f"conditional_mc_results.json (v{conditional_mc['version']}):")
for _scope_label in ('mixed_use', 'inference_dominant'):
    _sf = conditional_mc['single_facility_500mw'][_scope_label]
    _ef = conditional_mc['empirical_fleet'][_scope_label]
    print(f"  Scope: {_scope_label}")
    print(f"    Single facility (500 MW): mean={_sf['mean_commit_depth']:.1%}  "
          f"P5={_sf['p5_commit_depth']:.1%}")
    print(f"    Empirical fleet:          mean={_ef['mean_commit_depth']:.1%}  "
          f"P5={_ef['p5_commit_depth']:.1%}")
print()
print(f"  Contention onset (by scope): {conditional_mc['contention_onset_gw']}")

cascade_parameters.json (v2.0):
  Mixed-use static commit:        16.9%
  Inference-dominant static commit: 27.5%

conditional_mc_results.json (v2.0):
  Scope: mixed_use
    Single facility (500 MW): mean=24.6%  P5=23.7%
    Empirical fleet:          mean=21.1%  P5=13.2%
  Scope: inference_dominant
    Single facility (500 MW): mean=40.0%  P5=38.5%
    Empirical fleet:          mean=33.6%  P5=21.1%

  Contention onset (by scope): {'mixed_use': None, 'inference_dominant': None}


### 0.3 Contract Sanity Checks

In [3]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 0-3: CONTRACT SANITY CHECKS
# ══════════════════════════════════════════════════════════════════════════════

# Sanity checks under reformed framework structure (schema v2.0)
assert 'spatial_cascade' in cascade_params and len(cascade_params['spatial_cascade']) == 6
assert 'deferral_cascade' in cascade_params and len(cascade_params['deferral_cascade']) == 3
assert 'dvfs_cascade' in cascade_params and len(cascade_params['dvfs_cascade']) == 2
assert 'scope_scenarios' in cascade_params

for _scope_label in ('mixed_use', 'inference_dominant'):
    _sf = conditional_mc['single_facility_500mw'][_scope_label]
    assert 0.05 < _sf['mean_commit_depth'] < 0.80, (
        f"Single facility mean depth ({_scope_label}) "
        f"{_sf['mean_commit_depth']:.1%} outside plausible range"
    )

assert len(conditional_mc['per_gw_sweep']['inference_dominant']) >= 5

# Build sweep DataFrames per scope
sweep_df_by_scope = {
    scope: pd.DataFrame(records)
    for scope, records in conditional_mc['per_gw_sweep'].items()
}
# Backward-compat alias: default to inference-dominant for headline reporting
sweep_df = sweep_df_by_scope['inference_dominant']

print(f"✓ Contract sanity checks passed (reformed framework, v2.0)")
print(f"  Spatial parameters:  {len(cascade_params['spatial_cascade'])}")
print(f"  Deferral parameters: {len(cascade_params['deferral_cascade'])}")
print(f"  DVFS parameters:     {len(cascade_params['dvfs_cascade'])}")
print(f"  Sweep fleet sizes:   {list(sweep_df['fleet_gw'].values)}")

✓ Contract sanity checks passed (reformed framework, v2.0)
  Spatial parameters:  6
  Deferral parameters: 3
  DVFS parameters:     2
  Sweep fleet sizes:   [np.float64(0.5), np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0), np.float64(6.0), np.float64(8.0), np.float64(10.0), np.float64(12.0), np.float64(15.0)]


## Part 1: Accredited Capacity (§6)

- **1.1** ELCC application and reference cases
- **1.2** Per-GW accredited MW curve

Translates conditional MC commitment depths into accredited MW using PJM's
92% ELCC for demand response resources (ref. 29). The commitment depth ×
fleet MW gives curtailable load; the ELCC derate gives accredited capacity
that enters the capacity market.

### 1.1 ELCC Application and Reference Cases

In [4]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 1-1: ACCREDITED MW FROM CONDITIONAL MC COMMITMENT DEPTH
# ══════════════════════════════════════════════════════════════════════════════
# §6 calculation under reformed framework:
#   committed_mw  = depth × fleet_mw       (depth is per facility MW; PUE
#                                            already absorbed in NB02 cascade)
#   accredited_mw = committed_mw × ELCC
# Computed under both scope scenarios.
# ══════════════════════════════════════════════════════════════════════════════

DR_ELCC = 0.92  # PJM DR class rating

ref_results_by_scope = {}

for _scope_label in ('mixed_use', 'inference_dominant'):
    _df = sweep_df_by_scope[_scope_label].copy()
    _df['committed_mw']  = _df['mean'] * _df['fleet_mw']
    _df['accredited_mw'] = _df['committed_mw'] * DR_ELCC
    # P5-based accreditation (alternative to mean-based)
    _df['committed_mw_p5']  = _df['p5'] * _df['fleet_mw']
    _df['accredited_mw_p5'] = _df['committed_mw_p5'] * DR_ELCC

    _ref = []
    for _gw in (1.0, 10.0):
        _row = _df[_df['fleet_gw'] == _gw].iloc[0]
        _ref.append({
            'scope':              _scope_label,
            'fleet_gw':           _gw,
            'fleet_mw':           _row['fleet_mw'],
            'mean_depth':         _row['mean'],
            'p5_depth':           _row['p5'],
            'committed_mw':       _row['committed_mw'],
            'accredited_mw':      _row['accredited_mw'],
            'committed_mw_p5':    _row['committed_mw_p5'],
            'accredited_mw_p5':   _row['accredited_mw_p5'],
        })
    ref_results_by_scope[_scope_label] = {'sweep_df': _df, 'ref_cases': _ref}

# Combined reference table
ref_df = pd.DataFrame([
    r for d in ref_results_by_scope.values() for r in d['ref_cases']
])

# Backward-compat aliases (default to inference-dominant)
sweep_df = ref_results_by_scope['inference_dominant']['sweep_df']

# ─── Print ──────────────────────────────────────────────────────────────────
print("ACCREDITED CAPACITY BY REFERENCE FLEET SIZE — both scopes")
print("=" * 80)
print(f"  ELCC (PJM DR class): {DR_ELCC:.0%}")
print()
print(f"  {'Scope':<22} | {'Fleet':>6} | {'Mean':>7} | {'P5':>7} | "
      f"{'Mean Acc MW':>12} | {'P5 Acc MW':>10}")
print("  " + "─" * 78)
for _, _r in ref_df.iterrows():
    print(f"  {_r['scope']:<22} | {_r['fleet_gw']:>4.0f}GW | "
          f"{_r['mean_depth']:>6.1%} | {_r['p5_depth']:>6.1%} | "
          f"{_r['accredited_mw']:>10,.0f} | {_r['accredited_mw_p5']:>9,.0f}")

print()
print(f"  Mean-vs-P5 accreditation gap (inference-dominant):")
_id_1gw  = ref_df[(ref_df['scope']=='inference_dominant') & (ref_df['fleet_gw']==1.0)].iloc[0]
_id_10gw = ref_df[(ref_df['scope']=='inference_dominant') & (ref_df['fleet_gw']==10.0)].iloc[0]
print(f"    1 GW:  mean {_id_1gw['accredited_mw']:.0f} MW vs "
      f"P5 {_id_1gw['accredited_mw_p5']:.0f} MW  "
      f"(gap = {_id_1gw['accredited_mw'] - _id_1gw['accredited_mw_p5']:.0f} MW)")
print(f"    10 GW: mean {_id_10gw['accredited_mw']:.0f} MW vs "
      f"P5 {_id_10gw['accredited_mw_p5']:.0f} MW  "
      f"(gap = {_id_10gw['accredited_mw'] - _id_10gw['accredited_mw_p5']:.0f} MW)")

ACCREDITED CAPACITY BY REFERENCE FLEET SIZE — both scopes
  ELCC (PJM DR class): 92%

  Scope                  |  Fleet |    Mean |      P5 |  Mean Acc MW |  P5 Acc MW
  ──────────────────────────────────────────────────────────────────────────────
  mixed_use              |    1GW |  24.6% |  23.6% |        226 |       217
  mixed_use              |   10GW |  24.1% |  16.1% |      2,216 |     1,483
  inference_dominant     |    1GW |  39.8% |  38.1% |        367 |       350
  inference_dominant     |   10GW |  38.5% |  25.2% |      3,546 |     2,322

  Mean-vs-P5 accreditation gap (inference-dominant):
    1 GW:  mean 367 MW vs P5 350 MW  (gap = 16 MW)
    10 GW: mean 3546 MW vs P5 2322 MW  (gap = 1224 MW)


## Part 2: Avoided Capacity Cost (§6)

- **2.1** E3 levelized CT cost application
- **2.2** CEJA sensitivity

Computes the annualized system cost that accredited spatial migration capacity
would displace, using the E3 2025 Resource Adequacy Study levelized CT cost
as the marginal capacity resource.

### 2.1 Avoided Capacity Cost

In [5]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 2-1: AVOIDED CAPACITY COST (E3 levelized CT)
# ══════════════════════════════════════════════════════════════════════════════
# §6: accredited_mw × $/MW-yr = annual avoided capacity procurement cost.
# Reported under both scope scenarios.
# ══════════════════════════════════════════════════════════════════════════════

E3_CT_LEVELIZED_COST = 180_000   # $/MW-yr (E3 2025 IL RA Study, Table 6-1)
E3_CT_LEVELIZED_CEJA = 205_000   # $/MW-yr (Illinois CEJA sensitivity)

# ─── Apply to reference cases (both scopes) ─────────────────────────────────
ref_df['annual_avoided_base'] = ref_df['accredited_mw'] * E3_CT_LEVELIZED_COST
ref_df['annual_avoided_ceja'] = ref_df['accredited_mw'] * E3_CT_LEVELIZED_CEJA
ref_df['annual_avoided_base_p5'] = ref_df['accredited_mw_p5'] * E3_CT_LEVELIZED_COST

# ─── Apply to full sweep (inference-dominant for backward-compat alias) ─────
sweep_df['annual_avoided_base'] = sweep_df['accredited_mw'] * E3_CT_LEVELIZED_COST
sweep_df['annual_avoided_ceja'] = sweep_df['accredited_mw'] * E3_CT_LEVELIZED_CEJA

# Apply to per-scope sweep DataFrames too
for _scope_label, _bundle in ref_results_by_scope.items():
    _df_ = _bundle['sweep_df']
    _df_['annual_avoided_base'] = _df_['accredited_mw'] * E3_CT_LEVELIZED_COST
    _df_['annual_avoided_ceja'] = _df_['accredited_mw'] * E3_CT_LEVELIZED_CEJA

# ─── Print ──────────────────────────────────────────────────────────────────
print("AVOIDED CAPACITY COST — both scopes")
print("=" * 80)
print(f"  E3 levelized CT cost:  ${E3_CT_LEVELIZED_COST:,}/MW-yr (base)")
print(f"  E3 CEJA sensitivity:   ${E3_CT_LEVELIZED_CEJA:,}/MW-yr")
print()
print(f"  {'Scope':<22} | {'Fleet':>6} | {'Acc MW':>8} | "
      f"{'Avoided base':>13} | {'Avoided CEJA':>13}")
print("  " + "─" * 76)
for _, _r in ref_df.iterrows():
    print(f"  {_r['scope']:<22} | {_r['fleet_gw']:>4.0f}GW | "
          f"{_r['accredited_mw']:>7,.0f} | "
          f"${_r['annual_avoided_base']/1e6:>10,.0f}M | "
          f"${_r['annual_avoided_ceja']/1e6:>10,.0f}M")

print()
print(f"  Contention onset (by scope): {conditional_mc['contention_onset_gw']}")
print(f"  Beyond spatial contention onset, per-MW avoided cost declines as ")
print(f"  spatial commitment depth saturates (deferral and DVFS unaffected).")

AVOIDED CAPACITY COST — both scopes
  E3 levelized CT cost:  $180,000/MW-yr (base)
  E3 CEJA sensitivity:   $205,000/MW-yr

  Scope                  |  Fleet |   Acc MW |  Avoided base |  Avoided CEJA
  ────────────────────────────────────────────────────────────────────────────
  mixed_use              |    1GW |     226 | $        41M | $        46M
  mixed_use              |   10GW |   2,216 | $       399M | $       454M
  inference_dominant     |    1GW |     367 | $        66M | $        75M
  inference_dominant     |   10GW |   3,546 | $       638M | $       727M

  Contention onset (by scope): {'mixed_use': None, 'inference_dominant': None}
  Beyond spatial contention onset, per-MW avoided cost declines as 
  spatial commitment depth saturates (deferral and DVFS unaffected).


## Part 3: Interconnection Acceleration NPV (§6)

- **3.1** Assumptions and NPV computation
- **3.2** Extended Data Table export

The primary economic incentive for spatial migration investment is not capacity market revenue but interconnection queue acceleration. If a flexible interconnection agreement (FIA) allows a facility to energize before network upgrades complete, the present value of avoided delay depends on foregone compute revenue during the waiting period.

### 3.1 IX Queue NPV Computation

In [6]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 3-1: IX QUEUE NPV — BEHAVIORAL INCENTIVE (REFORMED FRAMEWORK)
# ══════════════════════════════════════════════════════════════════════════════
# IX acceleration NPV vs. capacity market revenue. Under the reformed framework
# there is no DVFS floor; the comparison is IX NPV vs. total commitment-depth
# capacity revenue (not a differential against a floor).
# ══════════════════════════════════════════════════════════════════════════════

# ─── Infrastructure constants (unchanged from prior) ────────────────────────
SYSTEM_MAX_POWER_KW = 10.2
GPUS_PER_SYSTEM     = 8
FACILITY_PUE        = 1.30
GPU_IT_POWER_KW     = SYSTEM_MAX_POWER_KW / GPUS_PER_SYSTEM
GPU_GRID_POWER_KW   = GPU_IT_POWER_KW * FACILITY_PUE
GPU_PER_MW_GRID     = int(1000 / GPU_GRID_POWER_KW)

GPU_RATE_HR          = 2.20
BRA_2027_28_PRICE    = 333.44     # $/MW-day (PJM 2027/28 BRA)
WACC                 = 0.10

# ─── From contracts (no hardcoded floor) ────────────────────────────────────
# 1 GW commitment depth from conditional MC per-GW sweep, inference-dominant
DEPTH_1GW_INF_DOM = ref_df.loc[
    (ref_df['scope'] == 'inference_dominant') & (ref_df['fleet_gw'] == 1.0),
    'mean_depth'
].iloc[0]
DEPTH_1GW_INF_DOM_P5 = ref_df.loc[
    (ref_df['scope'] == 'inference_dominant') & (ref_df['fleet_gw'] == 1.0),
    'p5_depth'
].iloc[0]

print("IX QUEUE NPV — BEHAVIORAL INCENTIVE QUANTIFICATION")
print("=" * 70)
print(f"  Reformed framework: no DVFS-only floor concept.")
print(f"  IX NPV compared against total commitment-depth cap revenue,")
print(f"  not a differential against a floor.")
print()

# ─── 1. Reference facility ──────────────────────────────────────────────────
FACILITY_GW    = 1.0
FACILITY_MW    = FACILITY_GW * 1000
UTILIZATION    = 0.80
HOURS_PER_YEAR = 8760

annual_compute_rev = (GPU_PER_MW_GRID * GPU_RATE_HR * HOURS_PER_YEAR
                      * UTILIZATION * FACILITY_MW)

print(f"Reference facility: {FACILITY_GW:.0f} GW grid-connected (inference-dominant)")
print(f"GPU density: {GPU_PER_MW_GRID:,} GPUs/MW (grid-metered, PUE={FACILITY_PUE})")
print(f"H100 spot rate: ${GPU_RATE_HR}/hr")
print(f"Utilization: {UTILIZATION:.0%}")
print(f"Annual gross compute revenue: ${annual_compute_rev/1e9:.2f}B/yr")
print()
print(f"Commitment depth (1 GW, inf-dominant): mean {DEPTH_1GW_INF_DOM:.1%}, "
      f"P5 {DEPTH_1GW_INF_DOM_P5:.1%}")
print()

# ─── 2. PJM IX queue baseline ───────────────────────────────────────────────
PJM_QUEUE_MEDIAN_YRS = 70 / 12
PJM_QUEUE_P75_YRS    = 84 / 12
print(f"PJM baseline queue [LBNL-2024]:")
print(f"  Median IR→COD: {PJM_QUEUE_MEDIAN_YRS:.1f} years")
print(f"  P75 IR→COD:    {PJM_QUEUE_P75_YRS:.1f} years")
print()

# ─── 3. FIA acceleration scenarios ──────────────────────────────────────────
acceleration_scenarios = {
    'Conservative': 2.0,
    'Central':      3.0,
    'Optimistic':   4.0,
}

# Annual capacity revenue at total commitment depth (mean), 1 GW inf-dominant
cap_rev_total_annual = (FACILITY_MW * DEPTH_1GW_INF_DOM
                         * BRA_2027_28_PRICE * 365 * DR_ELCC)

ix_npv_results = {}
print(f"WACC: {WACC:.0%} (hyperscaler range 8-12%)")
print()
print("─" * 78)
print(f"NPV OF IX QUEUE ACCELERATION ({FACILITY_GW:.0f} GW facility, inf-dominant)")
print("─" * 78)
print(f"{'Scenario':>14} | {'Accel (yrs)':>11} | {'Annuity':>8} | "
      f"{'NPV (gross)':>12} | {'vs Tot Cap Rev':>14}")
print("─" * 78)

for label, N in acceleration_scenarios.items():
    annuity = (1 - (1 + WACC)**(-N)) / WACC
    npv_gross = annual_compute_rev * annuity
    vs_cap_rev = npv_gross / cap_rev_total_annual
    ix_npv_results[label] = {'N': N, 'annuity': annuity, 'npv': npv_gross}
    print(f"{label:>14} | {N:>11.0f} | {annuity:>8.3f} | "
          f"${npv_gross/1e9:>10.1f}B | {vs_cap_rev:>12.0f}×")
print()

# ─── 4. Dominance summary ───────────────────────────────────────────────────
npv_central = ix_npv_results['Central']['npv']

print("─" * 78)
print("DOMINANCE SUMMARY")
print("─" * 78)
print(f"  Annual capacity revenue (1 GW inf-dominant, mean depth):")
print(f"    {DEPTH_1GW_INF_DOM:.1%} × {FACILITY_MW} MW × ${BRA_2027_28_PRICE}/MW-day "
      f"× 365 × {DR_ELCC:.0%} = ${cap_rev_total_annual/1e6:.1f}M/yr")
print()
print(f"  Central scenario IX NPV (3yr): ${npv_central/1e9:.1f}B")
print(f"  IX NPV / annual cap revenue:   {npv_central/cap_rev_total_annual:.0f}×")
print()
print(f"  Interpretation: even using mean (not differential) commitment depth,")
print(f"  IX acceleration NPV exceeds capacity-market revenue by ~2 orders of")
print(f"  magnitude. The qualitative finding from the prior framework holds:")
print(f"  capacity is the compliance hook, IX speed is the actual incentive.")

# ─── Store results ───────────────────────────────────────────────────────────
IX_NPV_CENTRAL     = ix_npv_results['Central']['npv']
IX_NPV_LOW         = ix_npv_results['Conservative']['npv']
IX_NPV_HIGH        = ix_npv_results['Optimistic']['npv']
IX_DOMINANCE_RATIO = npv_central / cap_rev_total_annual

IX QUEUE NPV — BEHAVIORAL INCENTIVE QUANTIFICATION
  Reformed framework: no DVFS-only floor concept.
  IX NPV compared against total commitment-depth cap revenue,
  not a differential against a floor.

Reference facility: 1 GW grid-connected (inference-dominant)
GPU density: 603 GPUs/MW (grid-metered, PUE=1.3)
H100 spot rate: $2.2/hr
Utilization: 80%
Annual gross compute revenue: $9.30B/yr

Commitment depth (1 GW, inf-dominant): mean 39.8%, P5 38.1%

PJM baseline queue [LBNL-2024]:
  Median IR→COD: 5.8 years
  P75 IR→COD:    7.0 years

WACC: 10% (hyperscaler range 8-12%)

──────────────────────────────────────────────────────────────────────────────
NPV OF IX QUEUE ACCELERATION (1 GW facility, inf-dominant)
──────────────────────────────────────────────────────────────────────────────
      Scenario | Accel (yrs) |  Annuity |  NPV (gross) | vs Tot Cap Rev
──────────────────────────────────────────────────────────────────────────────
  Conservative |           2 |    1.736 | $      16.1

### 3.2 Extended Data Table — Assumptions and Parameters

Exports the full set of input assumptions used in the IX acceleration NPV calculation (facility size, GPU density, compute revenue rate, discount rate, acceleration window, central NPV output) as a reference CSV for reviewer audit. Feeds the Extended Data section of the NE submission.

In [7]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 3-2: EXTENDED DATA TABLE — IX ACCELERATION ASSUMPTIONS
# ══════════════════════════════════════════════════════════════════════════════

import pandas as pd

ix_assumptions = [
    ('GPU_PER_MW_GRID', GPU_PER_MW_GRID, 'GPUs per MW of grid capacity (derived: 10.2 kW / 8 GPUs × 1.3 PUE)'),
    ('GPU_RATE_HR', GPU_RATE_HR, 'H100 spot rate, USD/hour (Jan 2026)'),
    ('WACC', WACC, 'Weighted average cost of capital'),
    ('ACCELERATION_YEARS_CENTRAL', 3, 'Queue acceleration window, central case (years)'),
    ('FACILITY_SIZE_MW', 1000, 'Reference facility size (MW)'),
    ('NPV_CONSERVATIVE_BILLIONS_USD', round(IX_NPV_LOW / 1e9, 1),
     'NPV of accelerated interconnection, conservative (2-year acceleration)'),
    ('NPV_CENTRAL_BILLIONS_USD', round(IX_NPV_CENTRAL / 1e9, 1),
     'NPV of accelerated interconnection, central (3-year acceleration)'),
    ('NPV_OPTIMISTIC_BILLIONS_USD', round(IX_NPV_HIGH / 1e9, 1),
     'NPV of accelerated interconnection, optimistic (4-year acceleration)'),
    ('DOMINANCE_RATIO', round(IX_DOMINANCE_RATIO, 1),
     'Ratio of IX NPV (central) to annual total capacity revenue at mean commitment depth'),
    ('DEPTH_1GW_INF_DOM_MEAN', round(DEPTH_1GW_INF_DOM, 3),
     'Mean commitment depth at 1 GW (inference-dominant scope) underlying NPV ratio'),
]

ix_assumptions_df = pd.DataFrame(ix_assumptions, columns=['Parameter', 'Value', 'Description'])
print(ix_assumptions_df.to_string(index=False))

tbl_path = REPO_ROOT / 'outputs' / 'tables' / 'extended_data_table_ix_acceleration_assumptions.csv'
ix_assumptions_df.to_csv(tbl_path, index=False)
print(f'Saved table: {tbl_path}')

                    Parameter    Value                                                                         Description
              GPU_PER_MW_GRID  603.000                  GPUs per MW of grid capacity (derived: 10.2 kW / 8 GPUs × 1.3 PUE)
                  GPU_RATE_HR    2.200                                                 H100 spot rate, USD/hour (Jan 2026)
                         WACC    0.100                                                    Weighted average cost of capital
   ACCELERATION_YEARS_CENTRAL    3.000                                     Queue acceleration window, central case (years)
             FACILITY_SIZE_MW 1000.000                                                        Reference facility size (MW)
NPV_CONSERVATIVE_BILLIONS_USD   16.100              NPV of accelerated interconnection, conservative (2-year acceleration)
     NPV_CENTRAL_BILLIONS_USD   23.100                   NPV of accelerated interconnection, central (3-year acceleration)
  NPV_OPTIMISTIC

## Part 4: Summary and Exports

- **4.1** Consolidated results table
- **4.2** Export final_results.json

### 4.1 Consolidated Results

In [8]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 4-1: CONSOLIDATED §6 RESULTS TABLE (both scopes)
# ══════════════════════════════════════════════════════════════════════════════

print("CONSOLIDATED §6 RESULTS — REFORMED FRAMEWORK")
print("=" * 80)
print()
print("ACCREDITED CAPACITY AND AVOIDED COST (both scopes)")
print("─" * 80)
for _, _r in ref_df.iterrows():
    print(f"  {_r['scope']} ({_r['fleet_gw']:.0f} GW):")
    print(f"    Mean commitment depth:   {_r['mean_depth']:.1%}  (P5: {_r['p5_depth']:.1%})")
    print(f"    Curtailable MW (mean):   {_r['committed_mw']:,.0f}")
    print(f"    Accredited MW (×{DR_ELCC:.0%}):  {_r['accredited_mw']:,.0f}  "
          f"(P5-based: {_r['accredited_mw_p5']:,.0f})")
    print(f"    Avoided cost (base):     ${_r['annual_avoided_base']/1e6:,.0f}M/yr")
    print(f"    Avoided cost (CEJA):     ${_r['annual_avoided_ceja']/1e6:,.0f}M/yr")
    print()

print("IX ACCELERATION NPV (1 GW facility, inference-dominant)")
print("─" * 80)
print(f"  Conservative (2yr): ${IX_NPV_LOW/1e9:.1f}B")
print(f"  Central (3yr):      ${IX_NPV_CENTRAL/1e9:.1f}B")
print(f"  Optimistic (4yr):   ${IX_NPV_HIGH/1e9:.1f}B")
print(f"  IX / annual total cap rev (mean depth): {IX_DOMINANCE_RATIO:.0f}×")
print()

print("CONTENTION AND SCALE DEPENDENCE")
print("─" * 80)
print(f"  Spatial contention onset by scope:")
for _scope_label, _onset in conditional_mc['contention_onset_gw'].items():
    print(f"    {_scope_label:<22}: {_onset} GW")
print(f"  Static commitment depth (Ensemble B central):")
print(f"    Mixed-use:           {cascade_params['static_commitment_depth_central']['mixed_use']:.1%}")
print(f"    Inference-dominant:  {cascade_params['static_commitment_depth_central']['inference_dominant']:.1%}")

# ─── Export summary CSV ──────────────────────────────────────────────────────
_summary_path = TABLES_DIR / 'system_implications_summary.csv'
sweep_df.to_csv(_summary_path, index=False)
print(f"\n  Wrote {_summary_path.name}")

# Also export the per-scope sweeps separately
for _scope_label, _bundle in ref_results_by_scope.items():
    _path = TABLES_DIR / f'system_implications_summary_{_scope_label}.csv'
    _bundle['sweep_df'].to_csv(_path, index=False)
    print(f"  Wrote {_path.name}")

CONSOLIDATED §6 RESULTS — REFORMED FRAMEWORK

ACCREDITED CAPACITY AND AVOIDED COST (both scopes)
────────────────────────────────────────────────────────────────────────────────
  mixed_use (1 GW):
    Mean commitment depth:   24.6%  (P5: 23.6%)
    Curtailable MW (mean):   246
    Accredited MW (×92%):  226  (P5-based: 217)
    Avoided cost (base):     $41M/yr
    Avoided cost (CEJA):     $46M/yr

  mixed_use (10 GW):
    Mean commitment depth:   24.1%  (P5: 16.1%)
    Curtailable MW (mean):   2,408
    Accredited MW (×92%):  2,216  (P5-based: 1,483)
    Avoided cost (base):     $399M/yr
    Avoided cost (CEJA):     $454M/yr

  inference_dominant (1 GW):
    Mean commitment depth:   39.8%  (P5: 38.1%)
    Curtailable MW (mean):   398
    Accredited MW (×92%):  367  (P5-based: 350)
    Avoided cost (base):     $66M/yr
    Avoided cost (CEJA):     $75M/yr

  inference_dominant (10 GW):
    Mean commitment depth:   38.5%  (P5: 25.2%)
    Curtailable MW (mean):   3,855
    Accredited MW (

### 4.2 Export Final Results

In [10]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 4-2: EXPORT final_results.json — REFORMED FRAMEWORK (v2.0)
# ══════════════════════════════════════════════════════════════════════════════

final_results = {
    'version': '2.0',
    'produced_by': 'notebook 03 — system_implications (reformed framework)',
    'elcc': DR_ELCC,
    'e3_ct_levelized_cost': E3_CT_LEVELIZED_COST,
    'e3_ct_levelized_ceja': E3_CT_LEVELIZED_CEJA,

    'reference_cases_by_scope': {
        scope: bundle['ref_cases']
        for scope, bundle in ref_results_by_scope.items()
    },

    'per_gw_curve_by_scope': {
        scope: bundle['sweep_df'][['fleet_gw', 'fleet_mw', 'mean', 'p5',
                                    'committed_mw', 'accredited_mw',
                                    'accredited_mw_p5',
                                    'annual_avoided_base',
                                    'annual_avoided_ceja']].to_dict(orient='records')
        for scope, bundle in ref_results_by_scope.items()
    },

    'ix_acceleration': {
        'facility_gw':           FACILITY_GW,
        'gpu_per_mw_grid':       GPU_PER_MW_GRID,
        'gpu_rate_hr':           GPU_RATE_HR,
        'annual_compute_rev':    annual_compute_rev,
        'wacc':                  WACC,
        'depth_1gw_inf_dom':     float(DEPTH_1GW_INF_DOM),
        'depth_1gw_inf_dom_p5':  float(DEPTH_1GW_INF_DOM_P5),
        'npv_conservative_2yr':  IX_NPV_LOW,
        'npv_central_3yr':       IX_NPV_CENTRAL,
        'npv_optimistic_4yr':    IX_NPV_HIGH,
        'dominance_ratio':       IX_DOMINANCE_RATIO,
        'comparison_base':       'annual capacity revenue at mean commitment depth (no floor differential)',
    },

    'contention_onset_gw_by_scope': conditional_mc['contention_onset_gw'],
}

_path = CONTRACTS_DIR / 'final_results.json'
with open(_path, 'w') as f:
    json.dump(final_results, f, indent=2, default=str)

print(f"✓ Wrote {_path.name}")
print(f"  Schema version:        {final_results['version']}")
print(f"  Reference cases (per scope, fleet GW): "
      f"{[(s, [r['fleet_gw'] for r in cases]) for s, cases in final_results['reference_cases_by_scope'].items()]}")
print(f"  IX NPV central:        ${IX_NPV_CENTRAL/1e9:.1f}B")

✓ Wrote final_results.json
  Schema version:        2.0
  Reference cases (per scope, fleet GW): [('mixed_use', [1.0, 10.0]), ('inference_dominant', [1.0, 10.0])]
  IX NPV central:        $23.1B
